# Patent Phrase Matching - DeBERTa Training
**For Google Colab**

⚠️ Runtime → Change runtime type → **A100 GPU** before running

### Current config
- DeBERTa-v3-large + BiLSTM + EMA + AWP(epoch 2+) + Target Groupby
- max_length=192, batch=4, epochs=3, AMP(autocast+GradScaler)
- **transformers==4.44.2 pinned** (newer versions break deberta-v3 training)

### Run order
1. Cell 1: Mount Drive
2. Cell 2: Check Drive space
3. Cell 3: Check data
4. Cell 4: Download script & patch paths
5. **Cell 4.5: Pin transformers==4.44.2 (required!)**
6. Cell 5: Start training
7. Cell 6 (optional): Clean up checkpoints

> If A100 disconnects, rerun Cell 4 → 4.5 → 5 only. **Never run Cell 6 during training** (it deletes in-progress checkpoints).

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/aicodinggym_2/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/aicodinggym_2/hf_cache',    exist_ok=True)
print('Directories ready')

In [ ]:
# Cell 2: Check Drive space
import subprocess

# Total Drive space
result = subprocess.run(['df', '-h', '/content/drive/MyDrive'], capture_output=True, text=True)
print('=== Drive total space ===', result.stdout)

# Per-file sizes under aicodinggym_2
ckpt_dir = '/content/drive/MyDrive/aicodinggym_2/checkpoints'
if os.path.exists(ckpt_dir):
    print('=== Checkpoint files ===')
    files = [(f, os.path.getsize(os.path.join(ckpt_dir, f))) for f in os.listdir(ckpt_dir)]
    files.sort(key=lambda x: x[1], reverse=True)
    total = 0
    for fname, size in files:
        print(f'  {fname}: {size/1e9:.2f} GB')
        total += size
    print(f'  total: {total/1e9:.2f} GB')
else:
    print('No checkpoint folder')

# HF cache size
hf_dir = '/content/drive/MyDrive/aicodinggym_2/hf_cache'
if os.path.exists(hf_dir):
    result2 = subprocess.run(['du', '-sh', hf_dir], capture_output=True, text=True)
    print(f'=== HF cache: {result2.stdout.split()[0]} ===')

In [ ]:
# Cell 3: Check data
data_path = '/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip'
if os.path.exists(data_path):
    size = os.path.getsize(data_path)
    print(f'Data found: {size/1e6:.1f} MB')
else:
    print('Data missing!')
    print(f'   path: {data_path}')
    print('   -> Upload the zip file to Drive')

In [ ]:
# Cell 4: Download script & patch paths
import re

!wget -q -O /content/deberta_finetune.py \
    'https://raw.githubusercontent.com/castlhoo/DSC204_us-patent-phrase-to-phrase-matching/main/deberta_finetune.py'

with open('/content/deberta_finetune.py', 'r') as f:
    code = f.read()

# Patch data path
code = re.sub(
    r'"data_path"\s*:\s*"data/us-patent-phrase-to-phrase-matching\.zip"',
    '"data_path"          : "/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip"',
    code
)
# Patch checkpoint path
code = re.sub(
    r'"ckpt_dir"\s*:\s*"checkpoints"',
    '"ckpt_dir"           : "/content/drive/MyDrive/aicodinggym_2/checkpoints"',
    code
)
# Store HF cache on Drive (avoids re-downloading after restart)
code = code.replace(
    'os.environ["HF_HOME"] = os.path.expanduser("~/.hf_cache")',
    'os.environ["HF_HOME"] = "/content/drive/MyDrive/aicodinggym_2/hf_cache"'
)

with open('/content/deberta_finetune.py', 'w') as f:
    f.write(code)

print('Paths patched')
!grep -n 'data_path\|ckpt_dir\|HF_HOME\|max_length\|batch_size\|epochs\|awp' /content/deberta_finetune.py | head -15

In [ ]:
# Cell 4.5: Pin transformers to match the UCSD DataHub environment
# Env that scored 0.858 on UCSD: transformers==4.44.2, torch==2.5.1+cu121
# Newer Colab transformers loads/trains deberta-v3 differently (val_pearson ~0.06).
# Cell 5 runs as a separate python process, so no runtime restart is needed.
!pip install -q transformers==4.44.2
import subprocess
r = subprocess.run(['python', '-c', 'import transformers, torch; print(transformers.__version__, torch.__version__)'],
                   capture_output=True, text=True)
print(f'versions: {r.stdout.strip()}')

In [ ]:
# Cell 5: Start training
!python /content/deberta_finetune.py

In [ ]:
# Cell 6 (optional): Clean up failed checkpoints
# Run ONLY when training ended abnormally and stale checkpoints remain on Drive.
# WARNING: do NOT run during training or mid-resume. It deletes in-progress fold checkpoints!
import os, glob

ckpt_dir = '/content/drive/MyDrive/aicodinggym_2/checkpoints'
# Delete only temp checkpoints of folds not yet marked completed in progress.json
import json
progress_path = os.path.join(ckpt_dir, 'progress.json')
completed = set()
if os.path.exists(progress_path):
    with open(progress_path) as f:
        completed = set(json.load(f).get('completed_folds', {}).keys())
print(f'Completed folds: {completed}')

removed = []
for fpath in glob.glob(os.path.join(ckpt_dir, 'fold*_step.pt')) + \
             glob.glob(os.path.join(ckpt_dir, 'fold*_latest.pt')) + \
             glob.glob(os.path.join(ckpt_dir, 'fold*_best.pt')):
    fname = os.path.basename(fpath)
    fold_num = fname.split('fold')[1].split('_')[0]
    if fold_num not in completed:  # only temp files of incomplete folds
        size = os.path.getsize(fpath)
        os.remove(fpath)
        removed.append(f'{fname} ({size/1e9:.2f} GB freed)')

if removed:
    print('Deleted files:')
    for r in removed:
        print(f'  {r}')
else:
    print('Nothing to delete (already clean)')